# Taller 2: Limpieza de Datos

**Gestión de Datos · Módulo 1 · 2026** — MSc. Edison Leonardo Neira Espitia

**Integrantes:** Luis Alberto Banda, Maria Camila Rincón, Juan Pablo Marin, Gabriel Camilo Velasquez y Juan Camilo Guerrero

**Entorno:** Docker con Jupyter Lab (imagen `quay.io/jupyter/scipy-notebook`); ver `Dockerfile`, `docker-compose.yml` y `requirements.txt`.

**Dataset:** Titanic — https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

## Ejercicio 1: Diagnóstico de calidad de los datos

### 4.1 Carga y vistazo inicial

Se importan las librerías y se carga `titanic.csv`. La primera vez se descarga desde la URL del enunciado y se guarda una copia local en `work/titanic.csv`, para que las siguientes ejecuciones no dependan de internet.

In [1]:
# Librerías del taller
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option("display.max_columns", None)

In [2]:
# Fuente del dataset y copia local
URL_TITANIC = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
RUTA_LOCAL = Path("titanic.csv")

# Descargar solo si no existe la copia local
if not RUTA_LOCAL.exists():
    pd.read_csv(URL_TITANIC).to_csv(RUTA_LOCAL, index=False)

df = pd.read_csv(RUTA_LOCAL)

# 10 primeras filas
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [3]:
# Dimensiones del dataset: (filas, columnas)
print("shape:", df.shape)

shape: (891, 12)


In [4]:
# Tipo de dato de cada columna
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

#### Reporte exploratorio con ydata-profiling

Se genera el reporte automático y se guarda en `work/titanic_profiling.html`. Sirve solo como apoyo: toda conclusión del diagnóstico se verifica con una celda propia más adelante.

Nota: `ydata_profiling` emite un `DeprecationWarning` al importarse (anuncia que el paquete cambiará de nombre). Se silencia solo ese aviso para que el notebook corra limpio; no afecta el reporte.

In [5]:
import warnings

# Silenciar únicamente el aviso de deprecación que emite ydata_profiling al importarse
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    from ydata_profiling import ProfileReport

# Reporte exploratorio del dataset original (sin limpiar)
perfil = ProfileReport(df, title="Titanic - Reporte exploratorio", explorative=True, progress_bar=False)

RUTA_REPORTE = Path("titanic_profiling.html")
perfil.to_file(RUTA_REPORTE)
print("Reporte guardado en:", RUTA_REPORTE.resolve())

  0%|          | 0/12 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:00<00:00, 175.57it/s]

Reporte guardado en: /home/jovyan/work/titanic_profiling.html


### 4.2 Tabla de diagnóstico por columna

Se construye una tabla con una fila por cada una de las 12 columnas y estas verificaciones:

- **Valores únicos o constantes:** columnas con un solo valor o con un valor distinto por fila (identificadores).
- **Valores faltantes:** número y % de nulos, y mecanismo probable (MCAR, MAR o MNAR).
- **Faltantes disfrazados:** ceros imposibles en `Age` y `Fare`, marcadores de texto (`"?"`, `"Unknown"`, `"NA"`, vacío, etc.) y tiquetes sin ningún dígito.
- **Valores atípicos por atributo:** límites de Tukey (Q1 − 1,5·IQR, Q3 + 1,5·IQR) para `Age`, `Fare`, `SibSp` y `Parch`.
- **Registros atípicos:** filas que combinan valores extremos en varios atributos, o con una tarifa atípica para su propia clase.
- **Redundancia o nulo valor:** correlación de Spearman (en valor absoluto) con la columna numérica más parecida, y comparación de `SibSp` y `Parch` frente al tamaño de familia.

La última columna de la tabla es la **acción de mitigación propuesta**. El mecanismo y la acción son un **borrador del agente** que el grupo debe revisar y corregir.

In [6]:
# Parámetros de las verificaciones
N_FILAS = len(df)
NUM_TUKEY = ["Age", "Fare", "SibSp", "Parch"]
COLS_CERO_IMPOSIBLE = ["Age", "Fare"]  # un 0 en edad o tarifa no es plausible
MARCADORES = {"?", "unknown", "na", "n/a", "none", "null", "-", ""}


def limites_tukey(serie):
    """Límites de Tukey: (Q1 - 1,5·IQR, Q3 + 1,5·IQR)."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def mascara_tukey(serie):
    """True donde el valor cae fuera de los límites de Tukey."""
    li, ls = limites_tukey(serie)
    return (serie < li) | (serie > ls)


def contar_disfrazados(col):
    """Cuenta faltantes disfrazados según el tipo de columna."""
    s = df[col].dropna()
    if col in COLS_CERO_IMPOSIBLE:
        return int((s == 0).sum()), "valor 0"
    if s.dtype == object:
        marcador = s.str.strip().str.lower().isin(MARCADORES)
        if col == "Ticket":
            sin_digitos = ~s.str.contains(r"\d")
            return int((marcador | sin_digitos).sum()), "marcador o ticket sin dígitos"
        return int(marcador.sum()), "marcador"
    return 0, "no aplica"


diagnostico = []
for col in df.columns:
    n_unicos = df[col].nunique(dropna=True)
    if n_unicos <= 1:
        tipo_unico = "Constante"
    elif n_unicos == N_FILAS:
        tipo_unico = "Un valor por fila"
    else:
        tipo_unico = "—"
    n_nulos = int(df[col].isna().sum())
    n_disfr, criterio = contar_disfrazados(col)
    fila = {
        "Columna": col,
        "dtype": str(df[col].dtype),
        "N únicos": n_unicos,
        "Único/constante": tipo_unico,
        "Nulos": n_nulos,
        "% nulos": round(100 * n_nulos / N_FILAS, 2),
        "Disfrazados": n_disfr,
        "Criterio disfrazado": criterio,
    }
    if col in NUM_TUKEY:
        li, ls = limites_tukey(df[col])
        n_atip = int(mascara_tukey(df[col]).sum())
        fila.update({
            "Tukey inf": round(li, 2),
            "Tukey sup": round(ls, 2),
            "Atípicos": n_atip,
            "% atípicos": round(100 * n_atip / N_FILAS, 2),
        })
    diagnostico.append(fila)

diagnostico = pd.DataFrame(diagnostico).set_index("Columna")
diagnostico

,dtype,N únicos,Único/constante,Nulos,% nulos,Disfrazados,Criterio disfrazado,Tukey inf,Tukey sup,Atípicos,% atípicos
Columna,,,,,,,,,,,
PassengerId,int64,891,Un valor por fila,0,0.00,0,no aplica,NaN,NaN,NaN,NaN
Survived,int64,2,—,0,0.00,0,no aplica,NaN,NaN,NaN,NaN
Pclass,int64,3,—,0,0.00,0,no aplica,NaN,NaN,NaN,NaN
Name,object,891,Un valor por fila,0,0.00,0,marcador,NaN,NaN,NaN,NaN
Sex,object,2,—,0,0.00,0,marcador,NaN,NaN,NaN,NaN
Age,float64,88,—,177,19.87,0,valor 0,-6.69,64.81,11.0,1.23
SibSp,int64,7,—,0,0.00,0,no aplica,-1.50,2.50,46.0,5.16
Parch,int64,7,—,0,0.00,0,no aplica,0.00,0.00,213.0,23.91
Ticket,object,681,—,0,0.00,4,marcador o ticket sin dígitos,NaN,NaN,NaN,NaN


#### Mecanismo de los faltantes

Para proponer el mecanismo se compara el % de nulos de `Age` y `Cabin` entre clases (`Pclass`) y entre sobrevivientes y no sobrevivientes (`Survived`). Si el % cambia mucho entre grupos, el faltante depende de otra variable observada (MAR) y no es completamente al azar. También se revisan las filas sin `Embarked`.

In [7]:
# % de nulos de Age y Cabin según Pclass y según Survived
for grupo in ["Pclass", "Survived"]:
    pct_nulos = df[["Age", "Cabin"]].isna().groupby(df[grupo]).mean().mul(100).round(1)
    print(f"% de nulos por {grupo}:")
    print(pct_nulos, "\n")

# Filas sin puerto de embarque
df.loc[df["Embarked"].isna()]

% de nulos por Pclass:
         Age  Cabin
Pclass             
1       13.9   18.5
2        6.0   91.3
3       27.7   97.6 

% de nulos por Survived:
           Age  Cabin
Survived             
0         22.8   87.6
1         15.2   60.2 



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


#### Registros atípicos

Se exploran filas completas de dos formas:

1. Cuántos atributos numéricos (`Age`, `Fare`, `SibSp`, `Parch`) tiene cada fila fuera de los límites de Tukey; las filas con 2 o más combinan valores extremos.
2. Tarifas atípicas **dentro de su propia clase** (Tukey calculado por `Pclass`), para encontrar casos como una tarifa altísima en tercera clase.

In [8]:
# 1) Número de atributos atípicos por fila
atipicos_por_attr = pd.DataFrame({col: mascara_tukey(df[col]) for col in NUM_TUKEY})
n_atip_fila = atipicos_por_attr.sum(axis=1)
print("Filas según número de atributos atípicos:")
print(n_atip_fila.value_counts().sort_index())

registros_multi = df.loc[n_atip_fila >= 2, ["PassengerId", "Pclass", "Age", "SibSp", "Parch", "Fare", "Survived"]]
print(f"\nFilas con 2 o más atributos atípicos: {len(registros_multi)}")
registros_multi.head(10)

Filas según número de atributos atípicos:
0    598
1    211
2     71
3     11
Name: count, dtype: int64

Filas con 2 o más atributos atípicos: 82


,PassengerId,Pclass,Age,SibSp,Parch,Fare,Survived
7,8,3,2.0,3,1,21.0750,0
16,17,3,2.0,4,1,29.1250,0
24,25,3,8.0,3,1,21.0750,0
27,28,1,19.0,3,2,263.0000,0
50,51,3,7.0,4,1,39.6875,0
54,55,1,65.0,0,1,61.9792,0
59,60,3,11.0,5,2,46.9000,0
63,64,3,4.0,3,2,27.9000,0
68,69,3,17.0,4,2,7.9250,1
71,72,3,16.0,5,2,46.9000,0


In [9]:
# 2) Tarifa atípica dentro de su propia clase
fare_atip_clase = df.groupby("Pclass")["Fare"].transform(mascara_tukey)
print("Tarifas atípicas dentro de su clase, por Pclass:")
print(df.loc[fare_atip_clase].groupby("Pclass").size())

# Las tarifas atípicas más altas de tercera clase
df.loc[fare_atip_clase & (df["Pclass"] == 3), ["PassengerId", "Pclass", "Fare", "Ticket", "SibSp", "Parch"]] \
    .sort_values("Fare", ascending=False).head(10)

Tarifas atípicas dentro de su clase, por Pclass:
Pclass
1    20
2     7
3    52
dtype: int64


,PassengerId,Pclass,Fare,Ticket,SibSp,Parch
201,202,3,69.5500,CA. 2343,8,2
180,181,3,69.5500,CA. 2343,8,2
159,160,3,69.5500,CA. 2343,8,2
324,325,3,69.5500,CA. 2343,8,2
792,793,3,69.5500,CA. 2343,8,2
863,864,3,69.5500,CA. 2343,8,2
846,847,3,69.5500,CA. 2343,8,2
692,693,3,56.4958,1601,0,0
169,170,3,56.4958,1601,0,0
643,644,3,56.4958,1601,0,0


#### Redundancia o nulo valor

Se calcula la correlación de Spearman entre las columnas numéricas (es robusta a atípicos y sirve para variables ordinales como `Pclass`). Además, se construye el tamaño de familia `FamilySize = SibSp + Parch + 1` para ver cuánto se solapa con `SibSp` y `Parch`.

In [10]:
# Correlación de Spearman (valor absoluto) entre columnas numéricas
corr_abs = df.select_dtypes("number").corr(method="spearman").abs()

# Para cada columna numérica, la otra columna con la que más se correlaciona
max_corr = {}
for col in corr_abs.columns:
    otra = corr_abs[col].drop(col).idxmax()
    max_corr[col] = f"{corr_abs.loc[col, otra]:.2f} ({otra})"

# SibSp y Parch frente al tamaño de familia
familia = df[["SibSp", "Parch"]].assign(FamilySize=df["SibSp"] + df["Parch"] + 1)
print("Spearman SibSp / Parch / FamilySize:")
print(familia.corr(method="spearman").round(2))

corr_abs.round(2)

Spearman SibSp / Parch / FamilySize:
            SibSp  Parch  FamilySize
SibSp        1.00   0.45        0.85
Parch        0.45   1.00        0.80
FamilySize   0.85   0.80        1.00


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.00,0.01,0.03,0.04,0.06,0.00,0.01
Survived,0.01,1.00,0.34,0.05,0.09,0.14,0.32
Pclass,0.03,0.34,1.00,0.36,0.04,0.02,0.69
Age,0.04,0.05,0.36,1.00,0.18,0.25,0.14
SibSp,0.06,0.09,0.04,0.18,1.00,0.45,0.45
Parch,0.00,0.14,0.02,0.25,0.45,1.00,0.41
Fare,0.01,0.32,0.69,0.14,0.45,0.41,1.00


#### Tabla resumen del diagnóstico

Se unen todas las verificaciones en una sola tabla. La columna **Registros atípicos** cuenta las filas en las que la columna es atípica y que además tienen 2 o más atributos atípicos (para `Fare` se añade cuántas tarifas son atípicas dentro de su clase).

**Borrador del agente, pendiente de revisión del grupo:** los textos de `MECANISMO` y `MITIGACION` en la siguiente celda los propuso el agente a partir de las salidas de arriba. El grupo debe revisarlos y editarlos antes de la entrega.

In [11]:
# BORRADOR DEL AGENTE: mecanismo probable de los faltantes (solo columnas con nulos)
MECANISMO = {
    "Age": "MAR: el % de nulos cambia según Pclass (27,7 % en 3.ª vs 6,0 % en 2.ª) y Survived",
    "Cabin": "MAR (posible MNAR): faltan 97,6 % en 3.ª clase vs 18,5 % en 1.ª",
    "Embarked": "MCAR: solo 2 filas, sin patrón visible (ambas de 1.ª clase, misma cabina B28)",
}

# BORRADOR DEL AGENTE: acción de mitigación propuesta para cada columna
MITIGACION = {
    "PassengerId": "Identificador: conservar como clave, excluir del análisis/modelado",
    "Survived": "Variable objetivo: sin cambios",
    "Pclass": "Sin cambios; se mantiene junto a Fare pese a rho = 0,69 (miden cosas distintas)",
    "Name": "Identificador: extraer el título (Mr, Mrs, Miss...) y no usar el nombre completo",
    "Sex": "Sin cambios; codificar como categórica",
    "Age": "Imputar con la mediana por grupo (Pclass y Sex) y agregar indicador de edad faltante; atípicos son edades plausibles: conservar",
    "SibSp": "Atípicos son familias grandes reales: conservar; resumir con FamilySize por redundancia",
    "Parch": "IQR = 0 marca como atípico todo valor distinto de 0: conservar; resumir con FamilySize",
    "Ticket": "Tratar 'LINE' como faltante; no usar como identificador (681 únicos en 891 filas)",
    "Fare": "Convertir los 15 ceros a NaN e imputar con la mediana por Pclass; atípicos reales: conservar o transformar con log",
    "Cabin": "Demasiados nulos para imputar: eliminar la columna y crear indicador TieneCabina (opcional: cubierta)",
    "Embarked": "Imputar con la moda (2 filas)",
}

# Registros atípicos: filas donde la columna es atípica y la fila tiene 2+ atributos atípicos
registros_atip = {}
for col in NUM_TUKEY:
    texto = f"{int((atipicos_por_attr[col] & (n_atip_fila >= 2)).sum())} filas con 2+ atípicos"
    if col == "Fare":
        texto += f"; {int(fare_atip_clase.sum())} atípicas dentro de su clase"
    registros_atip[col] = texto

tabla_diagnostico = diagnostico.assign(
    **{
        "Mecanismo": pd.Series(MECANISMO),
        "Registros atípicos": pd.Series(registros_atip),
        "Max |rho| (con)": pd.Series(max_corr),
        "Acción de mitigación": pd.Series(MITIGACION),
    }
)

# Orden de columnas según las verificaciones; la acción queda al final
tabla_diagnostico = tabla_diagnostico[[
    "dtype", "N únicos", "Único/constante",
    "Nulos", "% nulos", "Mecanismo",
    "Disfrazados", "Criterio disfrazado",
    "Tukey inf", "Tukey sup", "Atípicos", "% atípicos",
    "Registros atípicos", "Max |rho| (con)",
    "Acción de mitigación",
]].fillna("—")

# Mostrar los textos largos completos
with pd.option_context("display.max_colwidth", None):
    display(tabla_diagnostico)

,dtype,N únicos,Único/constante,Nulos,% nulos,Mecanismo,Disfrazados,Criterio disfrazado,Tukey inf,Tukey sup,Atípicos,% atípicos,Registros atípicos,Max |rho| (con),Acción de mitigación
Columna,,,,,,,,,,,,,,,
PassengerId,int64,891,Un valor por fila,0,0.00,—,0,no aplica,—,—,—,—,—,0.06 (SibSp),"Identificador: conservar como clave, excluir del análisis/modelado"
Survived,int64,2,—,0,0.00,—,0,no aplica,—,—,—,—,—,0.34 (Pclass),Variable objetivo: sin cambios
Pclass,int64,3,—,0,0.00,—,0,no aplica,—,—,—,—,—,0.69 (Fare),"Sin cambios; se mantiene junto a Fare pese a rho = 0,69 (miden cosas distintas)"
Name,object,891,Un valor por fila,0,0.00,—,0,marcador,—,—,—,—,—,—,"Identificador: extraer el título (Mr, Mrs, Miss...) y no usar el nombre completo"
Sex,object,2,—,0,0.00,—,0,marcador,—,—,—,—,—,—,Sin cambios; codificar como categórica
Age,float64,88,—,177,19.87,"MAR: el % de nulos cambia según Pclass (27,7 % en 3.ª vs 6,0 % en 2.ª) y Survived",0,valor 0,-6.69,64.81,11.0,1.23,2 filas con 2+ atípicos,0.36 (Pclass),Imputar con la mediana por grupo (Pclass y Sex) y agregar indicador de edad faltante; atípicos son edades plausibles: conservar
SibSp,int64,7,—,0,0.00,—,0,no aplica,-1.5,2.5,46.0,5.16,44 filas con 2+ atípicos,0.45 (Parch),Atípicos son familias grandes reales: conservar; resumir con FamilySize por redundancia
Parch,int64,7,—,0,0.00,—,0,no aplica,0.0,0.0,213.0,23.91,82 filas con 2+ atípicos,0.45 (SibSp),IQR = 0 marca como atípico todo valor distinto de 0: conservar; resumir con FamilySize
Ticket,object,681,—,0,0.00,—,4,marcador o ticket sin dígitos,—,—,—,—,—,—,Tratar 'LINE' como faltante; no usar como identificador (681 únicos en 891 filas)
